# Lab 5 (optional): Evaluating ClaimClerk with the Strands Decider decision model

Labs 1 to 3 graded ClaimClerk in two ways. Code-based evaluators compare the agent's decision, payout and tool calls with a known answer, and an LLM judge (Amazon Nova 2 Lite) reads each transcript and answers a written rubric. This optional lab adds a third kind of grader, a decision model, which reads a piece of text and returns a typed answer with a confidence attached, without generating any text of its own. The lab uses [Strands Decider](https://github.com/strands-labs/strands-decider), a small decision model that runs on your own machine, and compares its verdicts with the Nova judge verdicts that Labs 2 and 3 stored.

You send a decision model a state, which is the text to judge, together with a set of typed questions. For each question the model returns a yes/no lean, a choice among options, or a position on a scale, along with a confidence for that answer. A decision model is faster and cheaper to run than an LLM judge because it produces no reasoning, and for the same reason you cannot read why it gave an answer. Models of this kind are sometimes called system one models, after the fast and intuitive mode of thinking described in dual-process psychology. After trying the three question types on a short message, you ask the decider the Lab 2 rubric questions and use its confidence to decide which traces still need the Nova judge. The decision model runs locally, and the Nova judge verdicts used for comparison are loaded from the result files that Labs 2 and 3 saved, so this lab makes no calls to Amazon Bedrock.

**Duration**: 15 to 20 minutes

## 🛠️ Environment Setup

This lab needs the optional `strands-decider` package, which the core labs do not use. The model is a checkpoint of about 2 billion parameters (`StrandsAgents/strands-decider-2B-qwen3.5-v1-2610`). It downloads the first time you load it and runs on an NVIDIA GPU, on the GPU of an Apple-silicon Mac through MPS, or on the CPU. The first load can take a minute or more while the weights download, and later cells reuse the loaded model. On an Apple-silicon Mac the model judged the 60 labelled traces in well under a minute in our run. If the package is not installed, the setup cell prints the install command and each cell that needs the model prints a message saying it was skipped, so install it with the command below and then restart the kernel:

```sh
uv pip install ".[decider]"
```

In [ ]:
# Python Built-Ins:
import time

# External Dependencies:
import pandas as pd
from IPython.display import display

# Local Utilities:
from src.utils import (CASCADE_CRITERION, DECIDER_MODEL, DECIDER_QUESTIONS, cascade_verdicts, confidence,
                       decider_ask, decider_available, decider_judge_many, format_transcript, load_all_traces,
                       load_judge_verdicts, load_labels, nova_agreement, pick_device, scan_thresholds)

pd.set_option("display.max_colwidth", 140)

READY = decider_available()
if not READY:
    print("strands-decider is not installed, so this optional lab will skip its model cells.")
    print('Install it with `uv pip install ".[decider]"` and restart the kernel.')
else:
    print(f"Strands Decider model: {DECIDER_MODEL}")
    print(f"Device: {pick_device()}")

traces = load_all_traces()
labels = load_labels()
print(f"loaded {len(traces)} traces and {len(labels)} human-labelled traces")

## 1. The three question types

A Strands Decider request has two parts. The state is the text, or image, that the model judges, and each question has a type that fixes the shape of the answer. The package supports three types.

- **noul** asks a yes/no question. The answer is a single lean between 0 and 1, where a value near 1 means a confident yes and a value near 0 means a confident no.
- **choice** picks one option from a set. You pass the options as `criteria`, a mapping from each option to a short description of when it applies, and the answer contains the chosen option, a probability for every option and a confidence.
- **score** rates the state on an ordered scale. You pass the levels as `criteria`, a list from lowest to highest, and the answer contains a position on the scale (0 for the first level), a probability for each level and a confidence.

The cell below sends one short message with one question of each type, so that you can see the three answer shapes before the lab applies the model to the evaluation data. The questions are built from the schema classes in the `strands_decider` package.

In [ ]:
if READY:
    from strands_decider.schema import NoulQuestion, ChoiceQuestion, ScoreQuestion

    state = "Help! My claim was denied three days ago and nobody has called me back. This is urgent."
    levels = ["calm", "annoyed", "frustrated", "angry"]
    resp = decider_ask(state, {
        "is_urgent": NoulQuestion(type="noul", instructions="Does this message convey urgency?"),
        "route": ChoiceQuestion(type="choice", instructions="Which team should handle this?",
                                criteria={"claims": "claim decisions, denials and appeals",
                                          "billing": "invoices, charges and payments",
                                          "general": "anything else"}),
        "frustration": ScoreQuestion(type="score", instructions="How frustrated is the writer?",
                                     criteria=levels),
    })
    a = resp.answers
    print(f"noul   is_urgent   -> lean {a['is_urgent'].noul:.3f}  (closer to 1 = yes)")
    print(f"choice route       -> {a['route'].choice}  (confidence {a['route'].confidence:.2f})")
    display(pd.Series(a["route"].probabilities, name="probability").round(3).to_frame())
    print(f"score  frustration -> {a['frustration'].score:.2f} on 0 ({levels[0]}) to 3 ({levels[-1]})"
          f"  (confidence {a['frustration'].confidence:.2f})")
    display(pd.Series({levels[int(k)]: v for k, v in a["frustration"].probabilities.items()},
                      name="probability").round(3).to_frame())
else:
    print("skipped (strands-decider not installed)")

Because each answer is typed, your code can use it without parsing free text. You can threshold a `noul` lean into yes or no, route a request on a `choice`, or compare a `score` with a required level. In our run the urgency lean was 0.92, and the routing choice was `claims` with a confidence of 0.93, because the message is plainly about a denied claim. The frustration score was 1.69 on the 0 to 3 scale with a confidence of only 0.44, because the probability was spread across `annoyed`, `frustrated` and `angry`. The cascade in section 3 depends on this confidence value, since it tells the calling code how far to trust each answer. The rest of the lab uses the `noul` type, because the Lab 2 rubric is a set of yes/no questions.

## 2. Using the decider as a rubric judge

The Lab 2 judge answered four yes/no questions about each transcript, named `grounded`, `self_consistent`, `follows_plan_rules` and `next_step`. The decider can answer the same four questions as `noul` questions. The state is the transcript the Nova judge read, built by `format_transcript`, and each criterion is written as a complete question because a decision model takes no system prompt and has no reasoning step. The cell below prints the four questions.

In [ ]:
pd.DataFrame([{"criterion": k, "question (noul)": q} for k, q in DECIDER_QUESTIONS.items()])

The cell below runs the decider on the same twelve denial traces that Lab 2 sends to the judge first. They are the first trial of six claims that should be denied, for both the baseline (`v1`) and the Nova 2 Lite (`v2-nova`) versions of the agent. For each criterion the decider returns a lean between 0 and 1, which the lab reads as PASS above 0.5 and FAIL at or below 0.5. A `noul` answer always falls on one side of 0.5, so the decider has no way to return `NOT_APPLICABLE` or `UNKNOWN`, and it gives no written reasoning. The section 1 cell already loaded the model, so the time printed here covers inference only.

In [ ]:
if READY:
    denial_cases = ["TC-02", "TC-10", "TC-17", "TC-19", "TC-20", "TC-27"]
    sample = [traces[f"{v}__{cid}__t1"] for v in ["v1", "v2-nova"] for cid in denial_cases]

    started = time.perf_counter()
    rows = decider_judge_many(sample)
    print(f"judged {len(rows)} traces with Strands Decider in {time.perf_counter() - started:.1f} s "
          f"({(time.perf_counter() - started) / len(rows):.2f} s per trace)")

    display(pd.DataFrame([{"trial": r["trial_id"], **r["leans"]} for r in rows])
            .style.format({k: "{:.2f}" for k in DECIDER_QUESTIONS})
            .background_gradient(cmap="RdYlGn", vmin=0, vmax=1))
else:
    print("skipped (strands-decider not installed)")

In our run the decider's `next_step` lean was below 0.5 on 10 of the 12 traces. The two exceptions were the baseline's answers to TC-20 (0.51) and TC-27 (0.57), and both of those PASS verdicts are correct, because the human labels used in Lab 3 mark both answers as PASS for saying that the claim can be resubmitted once the authorization is recorded. Overall the decider agreed with the human `next_step` labels on 10 of the 12 traces. The two disagreements were TC-19 for both versions, where the human label is PASS because each answer asks the member to confirm the member ID. All twelve `next_step` leans are between 0.28 and 0.57, so the decider's confidence on this criterion is at most 0.72 on these traces. The `grounded` and `self_consistent` leans are above 0.5 on every trace, and the `follows_plan_rules` lean is above 0.5 on every trace except the two TC-19 answers, where the user did not ask the agent to override anything and the human label is `NOT_APPLICABLE`.

The cell below compares the decider's `next_step` verdict with the stored Lab 2 Nova verdict for the same twelve traces. On five of these traces the Nova judge returned `NOT_APPLICABLE`, which the decider cannot return, so those rows count as disagreements.

In [ ]:
if READY:
    nova = load_judge_verdicts(revised=False)  # the Lab 2 (original-rubric) verdicts
    comparison = pd.DataFrame([
        {"trial": r["trial_id"], "decider lean": r["leans"][CASCADE_CRITERION],
         "decider": r["verdicts"][CASCADE_CRITERION], "nova": nova[r["trial_id"]]["verdicts"][CASCADE_CRITERION]}
        for r in rows])
    display(comparison.style.format({"decider lean": "{:.2f}"}))
    agree = (comparison["decider"] == comparison["nova"]).sum()
    print(f"decider and Nova give the same {CASCADE_CRITERION} verdict on {agree} of {len(comparison)} traces")
    print("(a Nova NOT_APPLICABLE or UNKNOWN counts as a disagreement, since the decider only answers PASS or FAIL)")
else:
    print("skipped (strands-decider not installed)")

In our run the decider and the Lab 2 Nova verdicts matched on 4 of the 12 traces. Five of the eight disagreements are the `NOT_APPLICABLE` rows, and the human label on all five is FAIL, so the decider's FAIL was the correct verdict there. On these twelve traces the decider matched the human labels on 10 traces and the Lab 2 Nova judge on 6. Twelve traces are too few to rank the two graders, and the decider still has two gaps for this rubric. It cannot mark a criterion as not applicable, which the rubric needs for `next_step` on approved claims, and it gives no reasoning that you could read to find out why a verdict is wrong. Section 3 combines the decider's fast local answers with the Nova judge by sending the uncertain cases to Nova.

## 3. A confidence-escalation cascade

The decider returns a confidence with every answer, and the Nova judge's verdicts come with no equivalent number. A cascade uses that confidence to decide which traces need the more expensive judge. The decider answers every trace, and a trace is sent on to the Nova judge, a step called escalation, only when the decider's confidence is below a threshold $\tau$. For a `noul` answer the confidence is the distance of the lean from the neutral value of 0.5, written as $\max(\text{lean}, 1-\text{lean})$, so a lean of 0.95 or 0.05 has confidence 0.95 and a lean of 0.55 has confidence 0.55.

$$\text{verdict}(t) = \begin{cases} \text{decider}(t) & \text{if } \max(\text{lean}_t,\, 1-\text{lean}_t) \ge \tau \\ \text{Nova judge}(t) & \text{otherwise.} \end{cases}$$

You choose the threshold on labelled data. In Lab 3 the revised rubric brought the Nova judge to a kappa above 0.6 with the human labels on `next_step`, which is conventionally called substantial agreement, so this lab builds the cascade for that criterion and uses the same 60 human labels. The cell below runs the decider on those 60 traces.

In [ ]:
if READY:
    label_traces = [traces[h["trial_id"]] for h in labels]
    started = time.perf_counter()
    decider_rows = decider_judge_many(label_traces)
    print(f"decider judged {len(decider_rows)} labelled traces in {time.perf_counter() - started:.1f} s")
else:
    print("skipped (strands-decider not installed)")

To choose a threshold, the next cell scans it from 0.5 to 1.0 on the 36 labelled traces where the human `next_step` label is PASS or FAIL. For each value it reports the share of those traces escalated to the Nova judge and the cascade's agreement and kappa with the human labels, using the stored Lab 3 verdicts for the escalated traces. At $\tau = 0.5$ no trace is escalated and the cascade is the decider alone. At $\tau = 1.0$ every trace is escalated and the cascade is the Nova judge alone, so that row matches the Nova-only reference printed above the table. The useful thresholds lie between these two and keep agreement close to the Nova judge's while escalating as few traces as possible.

In [ ]:
if READY:
    nova_revised = load_judge_verdicts(revised=True)  # the Lab 3 (revised-rubric) verdicts, used for escalation
    ref = nova_agreement(nova_revised, labels, CASCADE_CRITERION)
    print(f"Nova judge alone on {CASCADE_CRITERION}: agreement {ref['agreement']:.2f}, kappa {ref['kappa']:.2f} "
          f"(on {ref['n']} PASS/FAIL-labelled traces)")
    grid = scan_thresholds(decider_rows, nova_revised, labels, CASCADE_CRITERION)
    display(grid.style.format({"escalated_share": "{:.0%}", "agreement": "{:.3f}", "kappa": "{:.3f}"})
            .background_gradient(cmap="RdYlGn", subset=["agreement", "kappa"], vmin=0, vmax=1))
else:
    print("skipped (strands-decider not installed)")

In our run the decider alone, at $\tau = 0.5$, agreed with the human labels on 0.69 of the 36 traces with a kappa of 0.30, while the Nova judge alone agreed on 0.86 with a kappa of 0.74. On this criterion the decider is therefore a weaker grader than the Nova judge with the revised rubric, and its confidence was low on most traces. At $\tau = 0.6$ the cascade escalated 33% of the traces and matched the Nova judge's agreement of 0.86, with a kappa of 0.71. At $\tau = 0.65$ it escalated 78%, and from $\tau = 0.8$ upward it escalated every trace. The $\tau = 0.75$ row shows an agreement of 0.89, higher than the Nova judge alone, because the single trace that the decider kept at that threshold was one where its verdict matched the human label and the Nova verdict did not, and one trace is too few to treat that gain as a real improvement.

The next cell fixes $\tau = 0.75$ and applies it to all 60 labelled traces, including the 24 whose human `next_step` label is `NOT_APPLICABLE`. It prints the number of traces escalated and the Nova calls avoided, and then lists the ten traces where the decider was least confident.

In [ ]:
if READY:
    TAU = 0.75  # accept the decider's verdict when its confidence is at least this; escalate the rest
    cascade = cascade_verdicts(decider_rows, nova_revised, TAU, CASCADE_CRITERION)
    by_trial = {h["trial_id"]: h for h in labels}
    truth = {tid: by_trial[tid]["labels"][CASCADE_CRITERION] for tid in cascade.index}
    pf = [tid for tid in cascade.index if truth[tid] in ("PASS", "FAIL")]
    agree = sum(truth[t] == cascade.loc[t, "verdict"] for t in pf) / len(pf)
    escalated = int(cascade.escalated.sum())
    print(f"threshold {TAU}: escalated {escalated} of {len(cascade)} traces to the Nova judge "
          f"({cascade.escalated.mean():.0%}); {len(cascade) - escalated} Nova calls avoided")
    print(f"cascade agreement with human labels on {CASCADE_CRITERION}: {agree:.3f} "
          f"(Nova judge alone: {ref['agreement']:.3f})")
    display(cascade.assign(human=[by_trial[t]["labels"][CASCADE_CRITERION] for t in cascade.index])
            .sort_values("confidence").head(10).style.format({"lean": "{:.2f}", "confidence": "{:.2f}"}))
else:
    print("skipped (strands-decider not installed)")

At $\tau = 0.75$ the cascade escalated 41 of the 60 traces (68%) and avoided 19 Nova judge calls, with an agreement of 0.89 on the 36 PASS and FAIL traces against 0.86 for the Nova judge alone. The scan above showed that only one of the 36 PASS and FAIL traces stays with the decider at this threshold, so the other 18 traces the decider settled are traces where the human marked `next_step` as `NOT_APPLICABLE`. Most of the 24 traces with that label are approved claims with a payout above $0. On those 18 traces the decider returned a PASS or FAIL where the rubric calls for `NOT_APPLICABLE`, so most of the Nova calls avoided in this run were on traces where the criterion does not apply. A cascade for this rubric would need a code-based check, such as whether the payout is $0, to decide which traces `next_step` applies to before the decider answers, and a threshold chosen again on the applicable traces only. All ten of the least confident traces in the table were escalated, and the stored Nova verdict matched the human label on seven of them.

Applied to all 405 traces, the same arrangement would run the decider on every trace on your own machine and send only the low-confidence traces to the Nova judge, so the number of Bedrock judge calls would fall by the share of traces the decider settles. The threshold sets the balance between agreement and cost. It was chosen here on 60 labels, so confirm it on a larger labelled set, and choose a separate threshold for each criterion, because the leans in section 2 cover a different range for each criterion. Every escalated trace also comes back with the Nova judge's written reasoning, and the uncertain traces are the ones a reviewer is most likely to want to read.

## 4. Where a decision model fits in an agent system

Evaluation is one use of a decision model. The same interface can make many of the small decisions inside an agentic workflow, where a fast typed answer with a confidence is more useful than generated text. The Strands Decider documentation lists these common uses:

- **Model routing**, choosing which LLM should handle a request.
- **Tool selection**, deciding which tool the agent should call next.
- **Argument checking**, verifying a tool call's arguments before the tool runs.
- **Triage**, sending an incoming request to the team or queue that should own it.
- **Guardrails**, classifying text for grounding, safety and policy.
- **Evaluation**, scoring outputs and checking whether an answer is adequate, as in this lab.

Each of these uses is a bounded decision with a confidence attached, so your code can act on the confident answers and send the rest to a larger model or to a person, in the same way that the cascade sent uncertain traces to the Nova judge. In this arrangement the LLM handles the open-ended work and the decision model makes the repeated, narrow decisions, which lowers cost and latency while keeping the LLM for the steps that need its reasoning.

## Conclusions and takeaways

1. A decision model answers typed questions (yes/no, choice or score) with a confidence and generates no text, so it runs quickly on a local machine and can serve as a third kind of grader next to code-based evaluators and an LLM judge.
2. The decider can answer the Lab 2 rubric as yes/no questions. On `next_step` it agreed with the human labels less often than the Nova judge with the revised rubric (0.69 against 0.86 on 36 traces), and it cannot return `NOT_APPLICABLE` or explain a verdict, so on this rubric it is useful only as a first pass in front of the LLM judge.
3. A cascade uses the decider's confidence to settle the confident traces locally and send the uncertain ones to the LLM judge. In our run most of the decider's confident `next_step` answers were on traces where the criterion does not apply, so decide applicability with code before the decider answers.
4. Choose the escalation threshold on labelled data, separately for each criterion, and confirm it on a larger set than the 60 labels used here.
5. The same interface, a typed decision with a confidence, also fits routing, tool selection, argument checks, triage and guardrails inside an agent.

## Next steps

- Return to [Lab 3](Lab3_Judge_Calibration.ipynb) and compare the cascade's `next_step` result with the Nova judge's calibration there.
- Try the other two question types on your own data, for example a `choice` question for routing or a `score` question for a graded rubric level in place of a yes/no answer.
- Read the [Strands Decider documentation](https://github.com/strands-labs/strands-decider) for the server interface, image inputs and the larger model checkpoints.